# AITuber — See-through 拆層（Colab GPU 版）

把一張角色立繪拆成 Live2D 用的分層 PSD（最多 23 層，含被遮住部分的 AI 補圖）。

**為什麼在 Colab 跑：** Mac mini M4 / 16GB 上 GPU（MPS）會報錯，只能用 CPU，記憶體峰值約 31GB，之前為了跑得動把參數降到解析度 512、1 步，結果不能用。這個筆記本用**官方預設品質**（解析度 1280、30 步）。

**使用方式：**
1. 上方選單「執行階段 → 變更執行階段類型 → T4 GPU」（有 L4/A100 更好）。
2. 「執行階段 → 全部執行」。
3. 第 3 格會跳出上傳按鈕，選你的立繪。
4. 跑完會自動下載 `see_through_<檔名>.zip`（內含 PSD、各層 PNG、預覽圖）。

**立繪建議：** 正面、手自然放下不碰身體、嘴閉眼開、翅膀尾巴不遮身體、純色背景、單一角色、無文字，長邊至少 1280px。

T4 大約 5–10 分鐘（第一次要下載模型）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, time
assert torch.cuda.is_available(), '沒有 GPU。請到「執行階段 → 變更執行階段類型」選 T4 GPU 後重新執行。'
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
CC = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {VRAM_GB:.1f} GB  capability: {CC}')
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('STEP 1/6 GPU_READY', time.strftime('%H:%M:%S'))

In [ ]:
# Clone See-through at a pinned commit and install inference dependencies.
# Colab already ships a CUDA build of torch, so the torch pins are skipped.
# Qt UI / training / dev-only packages are skipped too (not needed for inference).
import os, re, subprocess, time
SEE_THROUGH_COMMIT = 'a25a5498e031dc7fe01232d05dadaf67736277fb'
REPO = '/content/see-through'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/shitagaki-lab/see-through {REPO}
!git -C {REPO} fetch -q --depth 1 origin {SEE_THROUGH_COMMIT} && git -C {REPO} checkout -q {SEE_THROUGH_COMMIT}
%cd {REPO}
!ln -sfn common/assets assets

SKIP = re.compile(r'^(torch|torchvision|torchaudio|PyQt6|PyQt6-Qt6|qtpy|ipykernel|pytest|Cython|wheel|lpips|convnext_perceptual_loss)\b', re.I)
kept = [l for l in open('requirements.txt').read().splitlines()
        if l.strip() and not l.lstrip().startswith('#') and not SKIP.match(l.strip())]
open('req_colab.txt', 'w').write('\n'.join(kept) + '\n')
print('installing', len(kept), 'requirement lines')
!pip install -q -r req_colab.txt
!pip install -q -r requirements-inference-bnb.txt
print('STEP 2/6 DEPENDENCIES_READY', time.strftime('%H:%M:%S'))

In [ ]:
# Upload the character image.
import os, time
from PIL import Image
from google.colab import files

os.makedirs('/content/input', exist_ok=True)
up = files.upload()
assert up, '沒有上傳檔案。重新執行這一格並選擇立繪。'
name = list(up.keys())[0]
src = os.path.join('/content/input', os.path.basename(name))
os.replace(name, src)

im = Image.open(src)
print('input:', src, im.size, im.mode)
if max(im.size) < 1280:
    print('⚠ 長邊小於 1280px：See-through 會把圖放大到 1280 再拆，細節（眼睛、嘴巴）會糊。建議換更大的圖。')
if im.mode in ('RGBA', 'LA') or 'transparency' in im.info:
    # Flatten transparency onto white; the model expects an opaque illustration.
    bg = Image.new('RGB', im.size, (255, 255, 255))
    bg.paste(im.convert('RGBA'), mask=im.convert('RGBA').split()[-1])
    src = os.path.splitext(src)[0] + '_flat.png'
    bg.save(src)
    print('透明背景已壓成白底：', src)
SRC = src
print('STEP 3/6 INPUT_READY', time.strftime('%H:%M:%S'))

In [ ]:
# Run See-through at official quality: resolution 1280, 30 steps.
# >= 20 GB VRAM (L4/A100): full bf16 model, no offload (best quality, fastest).
# < 20 GB (T4): NF4 4-bit weights + group offload (~8 GB peak; quality close to bf16).
import time
RESOLUTION = 1280   # 官方預設；不要調低
STEPS = 30          # 官方預設；不要調低
SEED = 42
OUT_DIR = '/content/see_through_out'

if VRAM_GB >= 20 and CC[0] >= 8:
    MODE_ARGS = '--quant_mode none --no_group_offload'
else:
    MODE_ARGS = '--quant_mode nf4'
print('mode:', MODE_ARGS)
t0 = time.time()
!python inference/scripts/inference_psd_quantized.py --srcp "{SRC}" --save_dir {OUT_DIR} --resolution {RESOLUTION} --num_inference_steps {STEPS} --seed {SEED} --save_to_psd --tblr_split {MODE_ARGS}
print(f'STEP 4/6 SEE_THROUGH_DONE in {(time.time()-t0)/60:.1f} min', time.strftime('%H:%M:%S'))

In [ ]:
# Preview: reconstruction vs input, and every layer on a checkerboard.
import glob, os, math
from PIL import Image, ImageDraw
from IPython.display import display

stem = os.path.splitext(os.path.basename(SRC))[0]
RUN = os.path.join(OUT_DIR, stem)
psds = glob.glob(os.path.join(OUT_DIR, '**', '*.psd'), recursive=True)
print('PSD:', psds or '（沒有產生 PSD，請往上看錯誤訊息）')

def checker(size, n=24):
    bg = Image.new('RGB', size, (235, 235, 235)); d = ImageDraw.Draw(bg)
    for y in range(0, size[1], n):
        for x in range((y // n % 2) * n, size[0], 2 * n):
            d.rectangle((x, y, x + n - 1, y + n - 1), fill=(200, 200, 200))
    return bg

layer_pngs = sorted(p for p in glob.glob(os.path.join(RUN, '**', '*.png'), recursive=True)
                    if Image.open(p).mode == 'RGBA')
print(len(layer_pngs), 'RGBA layers')
tiles = []
for p in layer_pngs:
    im = Image.open(p).convert('RGBA'); bb = im.getbbox()
    if not bb: continue
    im = im.crop(bb); im.thumbnail((256, 256))
    t = checker((260, 280)); t.paste(im, ((260 - im.width) // 2, (260 - im.height) // 2), im)
    ImageDraw.Draw(t).text((4, 264), os.path.relpath(p, RUN)[:40], fill=(0, 0, 0))
    tiles.append(t)
if tiles:
    cols = 6; rows = math.ceil(len(tiles) / cols)
    sheet = Image.new('RGB', (cols * 260, rows * 280), (255, 255, 255))
    for i, t in enumerate(tiles):
        sheet.paste(t, ((i % cols) * 260, (i // cols) * 280))
    sheet.save(os.path.join(RUN, 'layers_contact_sheet.jpg'), quality=90)
    display(sheet)
rec = os.path.join(RUN, 'reconstruction.png')
if os.path.exists(rec):
    a = Image.open(SRC).convert('RGB'); b = Image.open(rec).convert('RGB').resize(a.size)
    pair = Image.new('RGB', (a.width * 2, a.height), 'white'); pair.paste(a, (0, 0)); pair.paste(b, (a.width, 0))
    pair.thumbnail((1400, 1400)); display(pair)
print('STEP 5/6 PREVIEW_READY')

In [ ]:
# Zip everything and download. Optional: also copy to Google Drive.
import shutil, os
from google.colab import files
SAVE_TO_DRIVE = False   # 改成 True 會同時存到 雲端硬碟/AITuber/see_through/
zip_base = f'/content/see_through_{stem}'
shutil.make_archive(zip_base, 'zip', RUN)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dst = '/content/drive/MyDrive/AITuber/see_through'; os.makedirs(dst, exist_ok=True)
    shutil.copy(zip_base + '.zip', dst); print('copied to', dst)
files.download(zip_base + '.zip')
print('STEP 6/6 DONE', zip_base + '.zip')

## 下一步（在 Mac 上）

1. 解壓後用 Krita / Photoshop 開 PSD，修補明顯錯誤（AI 補的遮擋區、左右分錯的零件）。
2. 把圖層匯出成同尺寸的全畫布 PNG，檔名加上繪製順序前綴（`01_hair_back.png`、`02_face.png`…），跑動作壓力檢查：

```bash
python3 "~/Bryce AI Studio/AITuber/source/tools/layer_qa/layer_motion_qa.py" <圖層資料夾>
```

3. 檢查 PASS 後再進 Cubism Editor 綁定。